# 10 SQL优化

## 八股问答

### 1. 插入数据怎么优化？

高频写入时不要逐条执行 `INSERT`，尽量合并成批量语句：

```sql
INSERT INTO orders (user_id, product_id, quantity, status)
VALUES
    (1, 10, 1, 'created'),
    (2, 20, 1, 'created'),
    (3, 30, 2, 'created');
```

需要处理重复记录时，可以用：

```sql
INSERT INTO orders (id, user_id, product_id, quantity, status)
VALUES (1, 1, 10, 1, 'created') AS new
ON DUPLICATE KEY UPDATE quantity = new.quantity;
```

对于大批量初始化或迁移，优先使用 `LOAD DATA`：

```sql
LOAD DATA INFILE '/data/orders.csv'
INTO TABLE orders
FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"'
LINES TERMINATED BY '\n'
IGNORE 1 LINES
(user_id, product_id, quantity, status);
```

其他注意点：

- 一次插入的数据量要合理，避免单条 SQL 过长或事务过大；
- 写入前检查索引数量，索引过多会拖慢插入；
- 大事务会长时间持有锁并放大 redo 日志，批量写入要控制事务边界；
- 使用 `INSERT ... ON DUPLICATE KEY UPDATE` 时，确认它真的执行了更新，而不是只做了一次无意义写入；
- 导入文件先校验字段顺序、编码、类型和重复数据，避免脏数据一次写入。

### 2. 主键怎么优化？

InnoDB 默认按主键组织聚簇索引，所以主键设计直接影响数据插入顺序、页分裂和范围查询。

推荐主键特性：

- 单调递增：新行尽量追加到索引末尾，减少页分裂；
- 尽量短：主键越短，二级索引叶子节点保存的主键越小；
- 非空且稳定：主键一旦确定，业务中不要频繁修改；
- 高选择性：不要把重复值很多或经常为空的字段直接当主键。

#### 页分裂

InnoDB 的数据页大小固定，默认约 16 KB。当一个叶子页已经写满，再插入新记录时，
需要把页拆成两个，并把部分记录移动到新页。页分裂会带来额外 I/O、页碎片和空闲空间，
随机主键会让插入位置分散在整个索引中，更容易频繁触发分裂；自增或雪花 ID 这类单调主键
通常只向索引末尾追加，页分裂较少。

#### 页合并

当大量记录被删除或更新离开原页后，叶子页的占用率可能明显下降，InnoDB 会把相邻的、
使用率很低的页合并，或复用页内空闲空间，从而减少碎片并回收空间。批量删除、归档业务
可能触发页合并，但频繁的删插交替也可能带来反复分裂与合并，所以大批量操作仍要分批、
控制事务大小。

单机或分库场景常用自增主键：

```sql
CREATE TABLE orders (
    id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
    user_id BIGINT NOT NULL,
    order_no VARCHAR(64) NOT NULL,
    created_at DATETIME NOT NULL,
    PRIMARY KEY (id),
    UNIQUE KEY uk_order_no (order_no)
) ENGINE=InnoDB;
```

#### 雪花 ID 简介

雪花 ID 通常是一个 64 位有符号整数，常见结构为：1 位符号位 + 41 位毫秒时间戳 +
10 位机器/工作节点 ID + 12 位同一毫秒内的递增序列。每个节点每毫秒理论上最多生成
4096 个 ID，不依赖中心数据库即可在分布式环境中保持唯一。

它的优点是整体按时间递增、长度短、适合作为 MySQL `BIGINT` 主键，也能减少随机插入
带来的页分裂。限制是依赖正确的工作节点 ID 和系统时钟；时钟回拨可能产生重复或乱序，
因此生产实现通常会在回拨时等待、使用备用时间戳或额外保存时钟偏移。雪花 ID 也不是
跨节点严格全局递增，只是每个节点内部基本有序。

分布式场景常用雪花 ID 或全局有序 ID，避免直接使用随机 UUID。随机 UUID 会破坏顺序，
容易导致页分裂和二级索引变大。

主键优化不是“必须自增”，而是先看业务是否允许单调 ID、是否需要跨库、是否要对外暴露主键。

### 3. ORDER BY 怎么优化？

`ORDER BY` 最好能直接利用索引顺序，避免 `Using filesort`。

#### Using filesort 是什么？

`Using filesort` 是 `EXPLAIN` 中 `Extra` 的提示，表示 MySQL 不能直接按索引顺序返回
结果，需要先把数据取出再额外排序；它可能用内存 `sort_buffer`，也可能用临时文件。
它不代表错误，是否优化要看扫描行数和实际耗时。

联合索引是否能直接服务排序，还要看排序列顺序和方向是否与索引一致。以
`idx_user_created(user_id, created_at)` 为例：

```sql
CREATE INDEX idx_user_created ON orders (user_id, created_at);

-- 和索引方向一致，能直接利用索引
SELECT id, order_no, created_at
FROM orders
ORDER BY user_id ASC, created_at ASC;

-- 两个排序列整体反向，也可以反向扫描索引
SELECT id, order_no, created_at
FROM orders
ORDER BY user_id DESC, created_at DESC;

-- 排序方向混合，通常需要 filesort
SELECT id, order_no, created_at
FROM orders
ORDER BY user_id ASC, created_at DESC;
```

常见减少 `filesort` 的方法：

- 让 `WHERE` 的等值列、`ORDER BY` 的排序列和索引列顺序匹配；
- 不要对排序列使用 `DATE()`、`LOWER()` 等函数；
- 排序方向尽量统一：`ASC` 与 `DESC` 混用有时会失去索引顺序；
- 只排真正需要的列，配合覆盖索引减少回表；
- 避免 `ORDER BY RAND()`，它会随机读取并排序大量数据。

### 4. GROUP BY 怎么优化？

`Using temporary` 是 `EXPLAIN` 中 `Extra` 的提示，表示 MySQL 需要创建临时表来协助
`GROUP BY`、`DISTINCT` 或排序等操作。

`GROUP BY` 如果无法使用索引，可能出现 `Using temporary` 或 `Using filesort`。

以 `idx_demo(province, age, status)` 为例：

```sql
CREATE INDEX idx_demo ON users (province, age, status);

-- 和索引最左列一致，可以直接按索引顺序扫描
SELECT province, COUNT(*)
FROM users
GROUP BY province;

-- 和索引前两列一致，也不需要临时表
SELECT province, age, COUNT(*)
FROM users
GROUP BY province, age;

-- status 不是最左列，同 status 的数据分散在不同 province/age 中，需要 Using temporary
SELECT status, COUNT(*)
FROM users
GROUP BY status;
```

为什么 `GROUP BY status` 会产生 `Using temporary`？因为索引按 `province → age → status`
排序，相同 `status` 的记录不会连续聚在一起，MySQL 无法直接按索引顺序完成分组，只能
把中间结果放入临时表再聚合；而 `GROUP BY province` 或 `GROUP BY province, age`
与索引前缀顺序一致，可以直接顺序扫描。

常见优化思路：

- 分组列尽量放联合索引左侧，并保持列顺序一致；
- 能用 `WHERE` 过滤的数据先过滤，避免分组大量无用数据；
- `HAVING` 不要承接本可以由 `WHERE` 完成的条件；
- 高基数分组结果很大时，考虑预聚合表或异步汇总；
- 需要精确计数时，再用 `COUNT(DISTINCT 列)`，并检查它是否真正需要。

### 5. LIMIT 怎么优化？

`LIMIT` 本身不一定是慢查询，慢的通常是 `ORDER BY ... LIMIT` 的深分页。

`LIMIT 50000, 20` 需要先扫描并丢弃前 50000 行：

```sql
-- 延迟关联：先取主键，再回表取完整行
SELECT o.*
FROM orders o
JOIN (
    SELECT id
    FROM orders
    ORDER BY id
    LIMIT 50000, 20
) t ON o.id = t.id;

-- 游标分页：根据上一页最后一条 id 继续取
SELECT *
FROM orders
WHERE id > 50000
ORDER BY id
LIMIT 20;
```

延迟关联的主要收益是：前面 50000 行只从较小的索引结构中读取主键，不读取完整行；
最后只对这 20 行回表取完整数据。因此它同时减少了跳过数据的读取量和回表次数，
而不是单纯优化回表。

注意事项：

- 排序字段必须有合适的索引，否则 `LIMIT` 仍会先排序全部数据；
- 游标分页要求排序字段唯一且稳定；
- 如果排序字段不唯一，加入其他唯一列作为排序条件；
- 返回列尽量精简，避免 `SELECT *` 带来不必要的回表。

### 6. UPDATE 怎么优化？

`UPDATE` 优化首先要保证 `WHERE` 能使用索引（也就是这个字段要有索引，否则会触发表锁，改不了），避免全表扫描后逐行修改：

```sql
UPDATE orders
SET status = 'archived'
WHERE status = 'done' AND id > 0
ORDER BY id
LIMIT 1000;
```




## 面试话术

“我遇到慢 SQL，会先看慢查询日志、执行频次和 EXPLAIN，确定是插入、排序、分组还是深分页
的问题。插入优先批量或 LOAD DATA，主键选择单调有序且尽量短；ORDER BY 和 GROUP BY
优先让索引匹配排列；深分页用延迟关联或游标；大表 UPDATE 按主键分批并控制事务。
改完再用 EXPLAIN ANALYZE 对比真实耗时和扫描行数。”


## 自查清单

- [ ] 能说明批量 INSERT、ON DUPLICATE KEY UPDATE 和 LOAD DATA 的适用场景
- [ ] 能解释自增主键、雪花 ID 和随机 UUID 的取舍
- [ ] 能说明 ORDER BY 如何配合索引避免 filesort
- [ ] 能说明 GROUP BY 如何减少 temporary 和 filesort
- [ ] 能写出 LIMIT 深分页的延迟关联和游标分页
- [ ] 能说明大表 UPDATE 为什么需要索引和分批
- [ ] 能按“优化前 EXPLAIN → 改写/加索引 → EXPLAIN ANALYZE → 上线观察”讲流程
